In [24]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.insert(0,"..")
from src.vocab.filtering import is_plausible_number_token
from src.vocab.filtering import build_plausible_vocab
from llm_sdk import Small_LLM_Model
from src.vocab.loader import loader_vocab
from src.fsm.value_matcher import allowed_token_ids_for_number
from src.generation.decoding import mask_logits, select_next_token

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
model = Small_LLM_Model()
dico_inverse = loader_vocab(model)
plausible_vocab = build_plausible_vocab(dico_inverse, is_plausible_number_token)
print(f"Number of plausible tokens: {len(plausible_vocab)}")
print(f"Number of total tokens: {len(dico_inverse)}")

Number of plausible tokens: 73
Number of total tokens: 151643


In [26]:
partial_value = ""
test = allowed_token_ids_for_number(partial_value, plausible_vocab)
print(f"Allowed token ids for '{partial_value}': {test}")
print(f"len allowed token ids: {len(test)}")

Allowed token ids for '': {1408, 513, 771, 85126, 12, 9229, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 53264, 27814, 18088, 5161, 97704, 98474, 16565, 3513, 313, 43449, 4421, 19528, 22349, 22738, 20948, 15081, 42091, 15340, 38645, 381}
len allowed token ids: 36


In [ ]:
def generate_number_value(
    full_prefix: str,
    plausible_vocab: dict[int, str],
    model: Small_LLM_Model)-> str:
    """
    Generate a number value that matches the allowed characters for a number token, given a full prefix.

    Args:
        full_prefix (str): The full prefix to be used for generating the number value.
        plausible_vocab (dict[int, str]): A dictionary mapping token ids to token texts.
        model (Small_LLM_Model): The language model to be used for generating the number value
    
    Returns:
        str: The generated number value that matches the allowed characters for a number token.
    """
    partial_value = ""
    while True:
        allowed_token_ids = allowed_token_ids_for_number(partial_value, plausible_vocab)
        full_text = full_prefix + partial_value
        input_ids = model.encode(full_text).tolist()[0]
        logits = model.get_logits_from_input_ids(input_ids)
        masked_logits = mask_logits(allowed_token_ids, logits)
        best_id = select_next_token(masked_logits)
        token_text = plausible_vocab.get(best_id, "")
        
        if "," in token_text or "}" in token_text :
            break
        
        partial_value += token_text
    return partial_value

In [34]:
full_prefix = '{"prompt": "What is the sum of 1201 and 53242?", "name": "fn_add_numbers", "parameters": {"a": '
value = generate_number_value(full_prefix, plausible_vocab, model)
print(f"Valeur générée : {value}")

1201 + , = 1201,
Valeur générée : 1201
